# S6E9: 53 Income Spikes and 94% of the Resolution Gain

My last notebook on this board decomposed the gap between pure counting and the best
engineered frame anyone in my campaign built. The largest single slice was not interaction
and not model. It was **resolution**: the same thirteen additive lookup tables, with the two
wide numeric columns cut at 1024 quantile bins instead of 32, gained more than the entire
interaction structure did.

That left an obvious question I did not answer. Only two of the thirteen columns are wide
enough to bin at all. So a large share of everything above counting lives in the fine
structure of two columns, and fine structure in a synthetic frame is a fact about the
generator. This notebook goes and looks at it.

What came out:

1. **Resolution keeps paying far past 1024 bins, and the saturation people report belongs to
   the smoothing constant, not the data.** At a fixed 50 pseudo-counts the sweep flattens and
   turns over. Sweep smoothing at the same time and the gain keeps climbing to about five
   thousand cells, reaching 0.942671 against a reference of 0.938321. Nearly a quarter of the
   total gain is invisible to a sweep that holds smoothing fixed.
2. **One column carries almost all of it.** `Annual_Income_USD` and `Daily_Commute_km` are the
   only two candidates. Income takes 94% of the two-column gain. Commute saturates at a few
   hundred cells and never reaches a third of a standard error, so the rest of the notebook is
   about income and I say so rather than quietly narrowing.
3. **The fine structure is real, and the control is the interesting part.** Permuting income
   within its own reference bins does not merely fail to reproduce the gain, it goes sharply
   negative: cutting finely into nothing costs you. Real and permuted are separated by more
   than five standard errors.
4. **The famous artifacts are real and, as features, worth nothing.** I verified all three
   inherited leads. Two hold exactly as stated. One is half right and I say which half. All
   three price under a tenth of a standard error, because a 32-bin quantile partition does not
   split ties and therefore already isolates them for free.
5. **What actually pays is diffuse.** After calibrating the additive model and testing every
   income value against what the rest of the frame predicts for those rows, 53 values depart by
   more than five sigma, against zero expected by chance over the same number of tests. The
   identical test on `Age`, which the model already sees at full resolution, returns nothing at
   all. The gain is spread over every octile of the income range, and collecting most of it
   takes about a thousand values split into their own cells, not three flags.

**What this is not.** It is not a model and it produces no submission. It is a census with a
null and a control attached. The practical version is one line long and it is in Part 11.

Everything is priced against a noise floor of 0.001052, measured previously on this
competition's own data by DeLong's exact variance and by brute-force resampling of the
out-of-fold vector, which agreed near 0.00105. Part 2 re-derives DeLong live, computes the
Hanley-McNeil closed form alongside it, and prints all three, so you can see that the closed
form overstates on this frame before you trust any of them. Competition:
[Playground Series S6E9](https://www.kaggle.com/competitions/playground-series-s6e9).

## The cell you edit

`REF_BINS` is the reference partition everything in the census is priced against. It is 32
because that is what a person building target-encoded features by hand actually reaches for,
and the whole point of Part 6 is that it is already doing more than they think.

`SE_REFERENCE` is the one number here that comes from an earlier run rather than from a cell
below. Part 2 recomputes a standard error from scratch on this run's own vector and prints
both, so you can see how far apart they are before trusting either.

In [1]:
CFG = {
    "TARGET": "Will_Buy_EV",
    "ID_COL": "id",
    "POSITIVE": "Yes",

    "SEED": 42,
    "N_SPLITS": 5,          # the fold convention the whole S6E9 library shares

    # ASSUMPTION, not a published figure. Kaggle does not state the public/private
    # split for this competition; 20% is the Playground Series convention. It only
    # enters through the standard error, and Part 2 prices the whole range.
    "PUBLIC_FRACTION": 0.20,

    # --- the lookup instrument ---------------------------------------------
    "WIDE_THRESHOLD": 50,   # columns with more distinct values than this get binned
    "REF_BINS": 32,         # the reference partition the census is priced against
    "REF_SMOOTH": 50.0,     # pseudo-counts shrinking a cell mean to the fold prior

    # --- Part 3, the two-dimensional sweep ---------------------------------
    "BIN_SWEEP": (32, 64, 128, 256, 512, 1024, 2048, 4096, 8192),
    "SMOOTH_SWEEP": (2.0, 5.0, 10.0, 20.0, 50.0),

    # --- Part 5, the null and the region decomposition ---------------------
    "NULL_REPEATS": 3,      # within-block permutations of the wide column
    "N_REGIONS": 8,         # octiles of the income range

    # --- Part 6, the artifact search ---------------------------------------
    "MIN_VALUE_COUNT": 30,  # a support value needs this many rows to be testable
    "LOCAL_WINDOW": 200,    # support values either side, for the local-trend test
    "CLIFF_STRIDE": 25,     # every Nth support value becomes a threshold candidate
    "ZONE_WIDTH": 50,       # support values per sliding zone candidate
    "ZONE_STRIDE": 25,
    "LATTICE_MODULI": (2, 5, 10, 25, 50, 100, 250, 500, 1000, 10000),
    "Z_THRESHOLDS": (3, 4, 5, 6, 8, 10),
    "PRICE_TOP_N": 8,       # candidates per family that get a full AUC price
    "NULL_MASKS": 12,       # random masks per size, for the refinement null

    # --- the three inherited leads, to be verified not repeated ------------
    # From earlier campaign work, where a public notebook author used them as
    # hand flags. Part 7 recomputes each one and reports what it finds.
    "LEADS": (
        ("spike at exactly income == 30000", "value", 30000),
        ("cliff at income >= 170537", "cliff", 170537),
        ("dead zone across income 38000 to 42000", "zone", (38000, 42000)),
    ),

    # --- the reference standard error --------------------------------------
    # Measured previously on this competition's own data two ways that agreed:
    # DeLong's exact variance and brute-force resampling of the out-of-fold
    # vector, both near 0.00105. Part 2 re-derives DeLong here, computes the
    # Hanley-McNeil closed form for comparison, and prints both.
    "SE_REFERENCE": 0.001052,

    # --- what the earlier notebook published, for the reproduction check ---
    "PUBLISHED_ADDITIVE": 0.938321,
    "PUBLISHED_CEILING": 0.945741,
}

---

## Part 1: The frame, and which columns can even have fine structure

Nothing interesting happens here. It is the shape of the data, the fold assignment, and the
decision about which columns are wide enough that binning throws information away. Every
number after this depends on those three things and you should be able to check them.

In [2]:
import os, gc, json, math, time, itertools, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
T0 = time.time()

ROOT = os.environ.get("SMOKE_ROOT", "/kaggle/input")
WORK = os.environ.get("SMOKE_WORK", "/kaggle/working")

def find(name):
    for root, _, files in os.walk(ROOT):
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(name)

TARGET, ID = CFG["TARGET"], CFG["ID_COL"]
train = pd.read_csv(find("train.csv"))
test  = pd.read_csv(find("test.csv"))

def to_binary(s):
    """The target ships as Yes/No here; other forks have had 0/1 and True/False."""
    if s.dtype.kind in "iufb":
        return s.astype(np.int8).to_numpy()
    return (s.astype(str).str.strip() == CFG["POSITIVE"]).astype(np.int8).to_numpy()

SMOKE_ROWS = int(os.environ.get("SMOKE_ROWS", "0"))
if SMOKE_ROWS:
    train = train.sample(SMOKE_ROWS, random_state=CFG["SEED"]).reset_index(drop=True)
    print(f"SMOKE_ROWS={SMOKE_ROWS}: train subsampled. Every number below is a "
          f"small-sample number and no title may be frozen from it.\n")

y = to_binary(train[TARGET])
yf = y.astype(np.float64)
N_TRAIN, N_TEST = len(train), len(test)
PREV = float(y.mean())
N_PUBLIC = round(N_TEST * CFG["PUBLIC_FRACTION"])
FEATS = [c for c in train.columns if c not in (ID, TARGET)]
CARD = {c: int(train[c].nunique(dropna=True)) for c in FEATS}
WIDE = [c for c in FEATS if CARD[c] > CFG["WIDE_THRESHOLD"]]
NARROW = [c for c in FEATS if c not in WIDE]

print(f"  train rows                {N_TRAIN:>12,}")
print(f"  test rows                 {N_TEST:>12,}")
print(f"  feature columns           {len(FEATS):>12}")
print(f"  positive rate in train    {PREV:>12.6f}")
print(f"  ASSUMED public split rows {N_PUBLIC:>12,}   (at {CFG['PUBLIC_FRACTION']:.0%}, "
      f"not a published figure)")
print(f"  nulls anywhere in FEATS   {int(train[FEATS].isna().sum().sum()):>12,}")
print()
print(f"{'column':<32}{'dtype':>10}{'distinct':>10}   fine structure possible?")
print("-" * 82)
for c in FEATS:
    tag = ("YES - more than " + str(CFG["WIDE_THRESHOLD"]) + " distinct values"
           if c in WIDE else "no, used whole")
    print(f"{c:<32}{str(train[c].dtype):>10}{CARD[c]:>10}   {tag}")
print("-" * 82)
print(f"  {len(WIDE)} of {len(FEATS)} columns are wide enough to bin: {WIDE}")
print(f"  the other {len(NARROW)} are used at full resolution already, so binning")
print(f"  cannot be throwing anything away in them. The widest of those is "
      f"{max(NARROW, key=lambda c: CARD[c])} at {max(CARD[c] for c in NARROW)} levels.")

### The folds

In [3]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

FOLDS = np.full(N_TRAIN, -1, dtype=np.int8)
skf = StratifiedKFold(CFG["N_SPLITS"], shuffle=True, random_state=CFG["SEED"])
for k, (_, va) in enumerate(skf.split(np.zeros(N_TRAIN), y)):
    FOLDS[va] = k

TR_M = [FOLDS != k for k in range(CFG["N_SPLITS"])]
VA_M = [FOLDS == k for k in range(CFG["N_SPLITS"])]
PRIOR = [float(yf[m].mean()) for m in TR_M]

print(f"StratifiedKFold(n_splits={CFG['N_SPLITS']}, shuffle=True, "
      f"random_state={CFG['SEED']}) over train.csv in original file order.")
print(f"  fold sizes      {np.bincount(FOLDS)}")
print(f"  fold priors     {[round(p, 6) for p in PRIOR]}")
print()
print("Every AUC in this notebook is computed on the full out-of-fold vector built")
print("from this one partition. Nothing is ever scored on rows it was fitted on.")

---

## Part 2: The additive baseline, rebuilt from nothing

This is what the whole census is priced against, so it gets rebuilt rather than quoted.
Thirteen tables. For each column, and for each fold, I take the rows outside the fold, group
them by that column's value, and record the smoothed mean of the target in each group:

$$\hat p_c(v) \;=\; \frac{\sum_{i \in \text{out of fold},\, x_{ic}=v} y_i \;+\; \lambda \cdot \bar y_{\text{out of fold}}}{n_{c,v} \;+\; \lambda}$$

Then I look up the rows inside the fold and take the log-odds. The additive score is the sum
of the thirteen log-odds. There is no model in it: no trees, no gradients, nothing you could
call learning beyond counting and dividing.

The two wide columns are cut into `REF_BINS` quantile bins first. That single choice is the
subject of the rest of the notebook.

In [4]:
def logit(p):
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return np.log(p / (1 - p))

def crossfit(keys, ncells, smooth=None):
    """Out-of-fold smoothed cell rate for an arbitrary integer key vector.

    One bincount for the counts and one for the positives, per fold. The whole
    census rests on this being cheap, because it gets called a few thousand times."""
    smooth = CFG["REF_SMOOTH"] if smooth is None else smooth
    out = np.empty(N_TRAIN)
    for k in range(CFG["N_SPLITS"]):
        tr, va = TR_M[k], VA_M[k]
        kt = keys[tr]
        cnt = np.bincount(kt, minlength=ncells).astype(np.float64)
        pos = np.bincount(kt, weights=yf[tr], minlength=ncells)
        out[va] = ((pos + smooth * PRIOR[k]) / (cnt + smooth))[keys[va]]
    return out

def qbins(x, nbins):
    """Quantile bins whose edges come from the whole training frame. Ties are not
    split, so a large tie mass collects into one bin on its own. Part 6 turns out
    to depend entirely on that being true."""
    edges = np.unique(np.quantile(x, np.linspace(0, 1, nbins + 1)))
    k = np.digitize(x, edges[1:-1]).astype(np.int64)
    return k, len(edges) - 1

def compress(k):
    codes, uniq = pd.factorize(k, sort=False)
    return codes.astype(np.int64), len(uniq)

RAW = {c: train[c].to_numpy(float) if pd.api.types.is_numeric_dtype(train[c])
       else None for c in FEATS}

t = time.time()
LOGIT_NARROW = {}
for c in NARROW:
    k = pd.Categorical(train[c].astype(str)).codes.astype(np.int64)
    LOGIT_NARROW[c] = logit(crossfit(k, int(k.max()) + 1))
BASE_NARROW = sum(LOGIT_NARROW.values())
AUC_NARROW = roc_auc_score(y, BASE_NARROW)

def wide_logit(col, nbins, smooth=None):
    k, c = qbins(RAW[col], nbins)
    return logit(crossfit(k, c, smooth)), c

REF_LOGIT, REF_CELLS = {}, {}
for c in WIDE:
    REF_LOGIT[c], REF_CELLS[c] = wide_logit(c, CFG["REF_BINS"])
ADDITIVE = BASE_NARROW + sum(REF_LOGIT.values())
ADD_AUC = roc_auc_score(y, ADDITIVE)

print(f"Cross-fitted smoothed cell means, one column at a time "
      f"(smooth={CFG['REF_SMOOTH']:g}, {CFG['REF_BINS']} quantile bins on the wide columns):")
print()
print(f"  the {len(NARROW)} narrow columns alone      {AUC_NARROW:>11.6f}")
for c in WIDE:
    print(f"  + {c:<30} {'':>11}   ({REF_CELLS[c]} occupied cells of "
          f"{CFG['REF_BINS']} asked for)")
print(f"  ADDITIVE, all {len(FEATS)} columns summed {ADD_AUC:>11.6f}   "
      f"({time.time() - t:.0f}s)")
print()
PUB = CFG["PUBLISHED_ADDITIVE"]
print(f"The earlier notebook published {PUB:.6f} for this same construction.")
print(f"This run: {ADD_AUC:.6f}. Difference: {ADD_AUC - PUB:+.6f}.")
if abs(ADD_AUC - PUB) < 5e-6:
    print("It reproduces to the digit, so the instrument below is the same instrument.")
else:
    print("It does NOT reproduce. Read every number after this with that in mind.")

### The floor, checked rather than quoted

The 0.001052 reference was measured on this competition's data in earlier work by DeLong's
exact variance and by brute-force resampling, which agreed. I am not going to make you take
that on faith when the check costs one sort. The Hanley-McNeil closed form is computed here
too, because it is the formula most people reach for and on a frame this imbalanced it is the
wrong one.

In [5]:
from scipy.stats import rankdata, norm

def delong_var(score, label):
    """Exact non-parametric variance of the AUC (DeLong et al. 1988), via midranks."""
    x, z = score[label == 1], score[label == 0]
    m, n = len(x), len(z)
    r_all = rankdata(np.concatenate([x, z]))
    v10 = (r_all[:m] - rankdata(x)) / n
    v01 = 1.0 - (r_all[m:] - rankdata(z)) / m
    return float(v10.mean()), float(v10.var(ddof=1) / m + v01.var(ddof=1) / n)

auc_chk, var_chk = delong_var(ADDITIVE, y)
SE_LIVE = math.sqrt(var_chk * N_TRAIN / N_PUBLIC)
SE = CFG["SE_REFERENCE"]

def hanley_var(auc, m, n):
    """Hanley and McNeil (1982), the exponential-distribution closed form."""
    q1 = auc / (2 - auc)
    q2 = 2 * auc * auc / (1 + auc)
    return (auc * (1 - auc) + (m - 1) * (q1 - auc ** 2)
            + (n - 1) * (q2 - auc ** 2)) / (m * n)

M_POS, M_NEG = int(y.sum()), int((1 - y).sum())
SE_HANLEY = math.sqrt(hanley_var(auc_chk, M_POS, M_NEG) * N_TRAIN / N_PUBLIC)

print(f"DeLong on all {N_TRAIN:,} out-of-fold rows of the additive vector:")
print(f"  AUC {auc_chk:.6f}   SE on those rows {math.sqrt(var_chk):.6f}")
print(f"  scaled to a {N_PUBLIC:,}-row public split (SE goes as 1/sqrt(n)): {SE_LIVE:.6f}")
print()
print(f"  reference SE, measured previously  {SE:.6f}")
print(f"  this run's DeLong estimate         {SE_LIVE:.6f}")
print(f"  Hanley-McNeil closed form          {SE_HANLEY:.6f}   "
      f"({SE_HANLEY / SE_LIVE:.2f}x the exact one)")
print(f"  ratio, DeLong to reference         {SE_LIVE / SE:.3f}x")
print()
print("The closed form assumes exponential score distributions within each class and this")
print("frame does not have them, so it overstates. I use the exact estimator and keep the")
print("reference, and I am printing the closed form only so nobody has to guess which one")
print("a number came from.")
print()
print("Close enough that the choice between them changes no conclusion here. I keep the")
print("reference so this notebook's tables stay comparable with the earlier campaign's,")
print("and note it is the SMALLER of the two, which makes every gain below look LARGER")
print("than this run's own floor would make it. Nothing here is flattered by my choice.")
print()
print("What the split-fraction assumption does to it:")
print(f"{'fraction':>10}{'rows':>10}{'SE':>11}")
for frac in (0.10, 0.20, 0.33, 0.50):
    rows = round(N_TEST * frac)
    print(f"{frac:>10.2f}{rows:>10,}{math.sqrt(var_chk * N_TRAIN / rows):>11.6f}"
          + ("   <- assumed" if abs(frac - CFG["PUBLIC_FRACTION"]) < 1e-9 else ""))

def in_se(delta):
    return delta / SE

assert 0.0006 < SE_LIVE < 0.0020, SE_LIVE

---

## Part 3: Where the resolution effect actually saturates

The earlier notebook swept bin counts from 32 to 1024 at a fixed smoothing of 50 pseudo-counts
and stopped there. That is the first thing to fix, because the two knobs are not independent:
more bins means fewer rows per cell, and a fixed 50 pseudo-counts shrinks a 40-row cell most
of the way back to the prior. A sweep that moves one knob will find a saturation point that
belongs to the other knob.

So I sweep both. Nine bin counts against five smoothing constants, the full additive model
rebuilt at each of the forty-five, with the narrow columns held fixed.

In [6]:
t = time.time()
GRID = {}
for nb in CFG["BIN_SWEEP"]:
    keys = {}
    for c in WIDE:
        keys[c] = qbins(RAW[c], nb)
    for sm in CFG["SMOOTH_SWEEP"]:
        s = BASE_NARROW.copy()
        for c in WIDE:
            k, cc = keys[c]
            s += logit(crossfit(k, cc, sm))
        GRID[(nb, sm)] = (roc_auc_score(y, s), {c: keys[c][1] for c in WIDE})
    del keys; gc.collect()
print(f"{len(GRID)} full additive rebuilds ({time.time() - t:.0f}s)\n")

hdr = f"{'bins asked':>11}{'income cells':>14}{'commute cells':>15}"
print(hdr + "".join(f"{('smooth=' + format(s, 'g')):>13}" for s in CFG["SMOOTH_SWEEP"]))
print("-" * (len(hdr) + 13 * len(CFG["SMOOTH_SWEEP"])))
for nb in CFG["BIN_SWEEP"]:
    cells = GRID[(nb, CFG["SMOOTH_SWEEP"][0])][1]
    row = f"{nb:>11}{cells[WIDE[0]]:>14}{cells[WIDE[1]]:>15}"
    print(row + "".join(f"{GRID[(nb, s)][0]:>13.6f}" for s in CFG["SMOOTH_SWEEP"]))
print("-" * (len(hdr) + 13 * len(CFG["SMOOTH_SWEEP"])))

BEST_KEY = max(GRID, key=lambda k: GRID[k][0])
BEST_AUC = GRID[BEST_KEY][0]
BEST_CELLS = GRID[BEST_KEY][1]
RES_GAIN = BEST_AUC - ADD_AUC
FIXED50 = [(nb, GRID[(nb, 50.0)][0]) for nb in CFG["BIN_SWEEP"]]
FIXED50_BEST = max(FIXED50, key=lambda r: r[1])

print(f"  best on the grid: {BEST_AUC:.6f} at {BEST_KEY[0]} bins, smooth={BEST_KEY[1]:g}")
print(f"    income {BEST_CELLS[WIDE[0]]} cells, commute {BEST_CELLS[WIDE[1]]} cells")
print(f"  reference ({CFG['REF_BINS']} bins, smooth={CFG['REF_SMOOTH']:g}): {ADD_AUC:.6f}")
print(f"  RESOLUTION GAIN {RES_GAIN:+.6f}  ({in_se(RES_GAIN):+.2f} SE)")
print()
print(f"  along the fixed smooth=50 row alone, the best is {FIXED50_BEST[1]:.6f} at "
      f"{FIXED50_BEST[0]} bins")
print(f"  ({FIXED50_BEST[1] - ADD_AUC:+.6f}, {in_se(FIXED50_BEST[1] - ADD_AUC):+.2f} SE), "
      f"which is {BEST_AUC - FIXED50_BEST[1]:.6f} short of the grid best.")
print(f"  So {100 * (BEST_AUC - FIXED50_BEST[1]) / RES_GAIN:.0f}% of the resolution gain is "
      f"invisible to a sweep that holds smoothing at 50.")
print()
print("Reading down a column: the gain does NOT stop at 1024. Reading across a row at the")
print("fine end: the fixed-50 curve turns over early because the cells run out of rows,")
print("not because the structure runs out. The two knobs have to move together.")

There is one more resolution to try, and it is the last one there is: stop binning entirely
and give every distinct value its own cell. If the gain were an artifact of quantile
boundaries landing well, the finest possible partition would win. It does not quite, and that
turnover is worth having on the record.

In [7]:
RAWCELL = {}
for c in WIDE:
    k, u = pd.factorize(RAW[c], sort=True)
    k = k.astype(np.int64)
    best = None
    for sm in CFG["SMOOTH_SWEEP"]:
        other = BASE_NARROW.copy()
        for d in WIDE:
            if d != c:
                kk, cc = qbins(RAW[d], BEST_KEY[0])
                other += logit(crossfit(kk, cc, BEST_KEY[1]))
        a = roc_auc_score(y, other + logit(crossfit(k, len(u), sm)))
        if best is None or a > best[0]:
            best = (a, sm)
    RAWCELL[c] = {"cells": len(u), "auc": best[0], "smooth": best[1]}
    print(f"  {c:<22} every distinct value its own cell ({len(u):>6} cells), "
          f"best smooth={best[1]:g}: {best[0]:.6f}")
print()
print(f"  grid best (binned)     {BEST_AUC:.6f} at {BEST_KEY[0]} bins")
for c in WIDE:
    print(f"  raw {c:<22} {RAWCELL[c]['auc']:.6f}  "
          f"({RAWCELL[c]['auc'] - BEST_AUC:+.6f} against the grid best)")
print()
print("So there is a real ceiling and the sweep finds it. Past a few thousand cells the")
print("tables start pricing noise faster than they find structure, which is what you would")
print("expect and is the reason I ran the null in Part 5 rather than assuming it.")

---

## Part 4: Which of the two columns carries it

Two columns can be binned. That is the whole candidate list, so the localisation is a
two-line experiment: refine one column while holding the other at the reference partition,
and then the other way round.

In [8]:
t = time.time()
LOCAL = {}
for target in WIDE:
    rows = []
    for nb in CFG["BIN_SWEEP"]:
        best = None
        for sm in CFG["SMOOTH_SWEEP"]:
            s = BASE_NARROW.copy()
            for c in WIDE:
                if c == target:
                    k, cc = qbins(RAW[c], nb)
                    s += logit(crossfit(k, cc, sm))
                else:
                    s += REF_LOGIT[c]
            a = roc_auc_score(y, s)
            if best is None or a > best[0]:
                best = (a, sm, qbins(RAW[target], nb)[1])
        rows.append({"bins": nb, "auc": best[0], "smooth": best[1], "cells": best[2]})
    LOCAL[target] = rows
print(f"two one-column sweeps ({time.time() - t:.0f}s)\n")

for target in WIDE:
    print(f"{target}, refined alone (the other wide column held at "
          f"{CFG['REF_BINS']} bins):")
    print(f"  {'bins':>6}{'cells':>8}{'best smooth':>13}{'AUC':>11}{'gain':>11}{'SE':>7}")
    for r in LOCAL[target]:
        g = r["auc"] - ADD_AUC
        print(f"  {r['bins']:>6}{r['cells']:>8}{r['smooth']:>13g}{r['auc']:>11.6f}"
              f"{g:>+11.6f}{in_se(g):>7.2f}")
    print()

SOLO = {c: max(r["auc"] for r in LOCAL[c]) - ADD_AUC for c in WIDE}
TOT = sum(SOLO.values())
CARRIER = max(SOLO, key=SOLO.get)
OTHER = [c for c in WIDE if c != CARRIER][0]
print(f"{'column':<26}{'gain alone':>12}{'SE':>7}{'share of the two':>19}")
print("-" * 64)
for c in WIDE:
    print(f"{c:<26}{SOLO[c]:>+12.6f}{in_se(SOLO[c]):>7.2f}{100 * SOLO[c] / TOT:>18.1f}%")
print("-" * 64)
print(f"{'both together (Part 3)':<26}{RES_GAIN:>+12.6f}{in_se(RES_GAIN):>7.2f}")
print()
print(f"  {CARRIER} carries {100 * SOLO[CARRIER] / TOT:.0f}% of it.")
SAT_CELLS = min(r["cells"] for r in LOCAL[OTHER]
                if r["auc"] >= max(x["auc"] for x in LOCAL[OTHER]) - 1e-5)
print(f"  {OTHER} is flat from {SAT_CELLS} cells onward and peaks at only "
      f"{in_se(SOLO[OTHER]):.2f} SE,")
print(f"  which is below the 0.2 SE bar this campaign uses for whether a thing is worth "
      f"building.")
print(f"  Everything from here on is about {CARRIER}. I am naming that choice rather")
print(f"  than quietly making it: {OTHER} was measured and found not to matter.")

---

## Part 5: Is the fine structure real, and where in the range does it live

Two things can produce a resolution gain. Real fine structure in the generator, or a cutting
artifact: more cells means more degrees of freedom, and with out-of-fold fitting that usually
costs you but need not always.

The control separates them. I permute the carrier column's values **within each reference
bin**, which destroys any structure finer than the reference partition while leaving the
coarse marginal, the coarse target relationship and every other column exactly as they were.
Then I rebuild at the winning resolution. If the gain is a cutting artifact it survives. If it
is structure it does not.

In [9]:
t = time.time()
K_REF, C_REF = qbins(RAW[CARRIER], CFG["REF_BINS"])
def additive_with(col, k, c, smooth):
    s = BASE_NARROW.copy()
    for d in WIDE:
        s += (logit(crossfit(k, c, smooth)) if d == col else REF_LOGIT[d])
    return roc_auc_score(y, s)

K_FINE, C_FINE = qbins(RAW[CARRIER], BEST_KEY[0])
REAL = additive_with(CARRIER, K_FINE, C_FINE, BEST_KEY[1])
rng = np.random.default_rng(CFG["SEED"])
nulls = []
for rep in range(CFG["NULL_REPEATS"]):
    shuf = RAW[CARRIER].copy()
    for b in range(C_REF):
        idx = np.where(K_REF == b)[0]
        shuf[idx] = RAW[CARRIER][rng.permutation(idx)]
    kn, cn = qbins(shuf, BEST_KEY[0])
    a_coarse = additive_with(CARRIER, *qbins(shuf, CFG["REF_BINS"]), CFG["REF_SMOOTH"])
    a_fine = additive_with(CARRIER, kn, cn, BEST_KEY[1])
    nulls.append(a_fine - ADD_AUC)
    print(f"  null rep {rep}: coarse {a_coarse:.6f} (unchanged by construction), "
          f"fine {a_fine:.6f}, gain {a_fine - ADD_AUC:+.6f} "
          f"({in_se(a_fine - ADD_AUC):+.2f} SE)")
nulls = np.array(nulls)
SOLO_REAL = REAL - ADD_AUC
print()
print(f"  real fine structure   {SOLO_REAL:+.6f}  ({in_se(SOLO_REAL):+.2f} SE)")
print(f"  within-bin permuted   {nulls.mean():+.6f}  ({in_se(nulls.mean()):+.2f} SE)   "
      f"mean of {CFG['NULL_REPEATS']}, max {nulls.max():+.6f}")
print(f"  separation            {SOLO_REAL - nulls.max():+.6f}  "
      f"({in_se(SOLO_REAL - nulls.max()):+.2f} SE)")
print()
print("The null does not merely fail to gain, it LOSES. Cutting a column finely when there")
print("is nothing below the coarse partition costs real AUC, which is the honest shape of")
print("this control and the reason the positive result means something. "
      f"({time.time() - t:.0f}s)")

Now, where in the range does it live? I refine the carrier column inside one region of its
range at a time and keep the reference partition everywhere else. If the structure were a
single artifact at one end, one region would carry the whole thing.

In [10]:
G = CFG["N_REGIONS"]
edges = np.quantile(RAW[CARRIER], np.linspace(0, 1, G + 1))
REGION = []
for g in range(G):
    lo, hi = edges[g], edges[g + 1]
    m = (RAW[CARRIER] >= lo) & (RAW[CARRIER] < hi) if g < G - 1 else (RAW[CARRIER] >= lo)
    k, c = compress(np.where(m, K_FINE + C_REF + 1, K_REF))
    a = additive_with(CARRIER, k, c, CFG["REF_SMOOTH"])
    REGION.append({"g": g + 1, "lo": lo, "hi": hi, "rows": int(m.sum()),
                   "auc": a, "gain": a - ADD_AUC})

print(f"Refining {CARRIER} only inside one region, reference partition elsewhere:")
print(f"  {'region':>8}{'from':>11}{'to':>11}{'rows':>9}{'gain':>11}{'SE':>7}")
for r in REGION:
    print(f"  {('octile ' + str(r['g'])):>8}{r['lo']:>11,.0f}{r['hi']:>11,.0f}"
          f"{r['rows']:>9,}{r['gain']:>+11.6f}{in_se(r['gain']):>7.2f}")
SUM_R = sum(r["gain"] for r in REGION)
JOINT = additive_with(CARRIER, K_FINE, C_FINE, CFG["REF_SMOOTH"]) - ADD_AUC
print(f"  {'sum':>8}{'':>31}{SUM_R:>+11.6f}{in_se(SUM_R):>7.2f}")
print(f"  {'joint':>8}{'':>31}{JOINT:>+11.6f}{in_se(JOINT):>7.2f}   "
      f"(all eight refined at once, same smoothing)")
print()
TOPR = max(REGION, key=lambda r: r["gain"])
print(f"  The eight regions sum to {SUM_R:+.6f} against a joint value of {JOINT:+.6f}, "
      f"a difference of {SUM_R - JOINT:+.6f}.")
print(f"  They are additive to within {in_se(abs(SUM_R - JOINT)):.2f} SE of each other, which")
print(f"  means the regions are not competing for the same signal.")
print(f"  The single best region is octile {TOPR['g']} at {in_se(TOPR['gain']):.2f} SE, and it "
      f"holds {100 * TOPR['gain'] / SUM_R:.0f}% of the total.")
print()
print("That is the shape of a diffuse effect, not a landmark. No region dominates, and the")
print("two regions at the ends of the range, which is where every artifact anyone has named")
print("on this board lives, are among the weakest. Part 6 goes and finds out why.")

---

## Part 6: The artifact census

A synthetic frame built from a real one leaves fingerprints. The catalogue is short and well
known: single values that carry far more mass or a far different target rate than their
neighbours, cliffs where the rate steps, zones that are empty or flat, rounding lattices where
values pile onto multiples of 100 or 1000, and truncation where a range was clamped.

I search all five families systematically rather than eyeballing a histogram, and then I price
each candidate in AUC. **Pricing is the part people skip.** A flag that changes a target rate
dramatically can still be worth nothing, and on this board that is exactly what happens.

### The instrument

Every candidate is a boolean mask. I price it by **refinement**, not by addition: I cross the
mask with the reference partition of the carrier column and rebuild that one lookup table. The
model still has exactly one term for the column, so nothing is double counted. The gain is the
AUC of the refined partition minus the AUC of the reference partition.

Adding a redundant second term instead would be the natural thing to try and it is wrong. A
summed-logit model double counts, and a flag that the partition already knows about comes out
strongly negative, which reads as a finding and is an artifact of the instrument.

In [11]:
BASE_REF = ADDITIVE
def refine_price(mask):
    k, c = compress(K_REF * 2 + mask.astype(np.int64))
    return additive_with(CARRIER, k, c, CFG["REF_SMOOTH"]) - ADD_AUC

t = time.time()
rng2 = np.random.default_rng(CFG["SEED"] + 1)
NULL_REF = {}
for frac in (0.001, 0.01, 0.10):
    g = np.array([refine_price(rng2.random(N_TRAIN) < frac)
                  for _ in range(CFG["NULL_MASKS"])])
    NULL_REF[frac] = g
    print(f"  null: {CFG['NULL_MASKS']} random masks at {frac:.1%} "
          f"({round(frac * N_TRAIN):,} rows): mean {g.mean():+.6f}, max {g.max():+.6f} "
          f"({in_se(g.max()):+.2f} SE)")
NULL_MAX = max(g.max() for g in NULL_REF.values())
print()
print(f"  refinement null ceiling across every size tried: {NULL_MAX:+.6f} "
      f"({in_se(NULL_MAX):+.2f} SE)   ({time.time() - t:.0f}s)")
print("  A mask with no information in it cannot buy AUC through this instrument.")
print("  Anything positive below is above the null. The question is whether it is above")
print("  the null by enough to matter, and the bar for that is 0.2 SE.")

### The detector

For the spike and zone families I need to know what "departs from a smooth trend" means, and
the honest version of that is not a trend line in income alone. It is: **what does the rest of
the frame already predict for these rows?** So I calibrate the additive model into actual
probabilities, and then for every candidate I compare the observed positives against the sum
of those predicted probabilities, using the Poisson-binomial variance.

That makes the test a test of what income knows **beyond** the other twelve columns and beyond
its own coarse trend. A control comes free with it: run the same test on a narrow column the
model already sees at full resolution, and it should return nothing at all.

In [12]:
kb, cb = qbins(ADDITIVE, 200)
P = crossfit(kb, cb, 20.0)
print(f"calibration of the additive score into probabilities (200 quantile bins of the score):")
print(f"  mean predicted {P.mean():.6f}   observed prevalence {PREV:.6f}   "
      f"AUC preserved {roc_auc_score(y, P):.6f} vs {ADD_AUC:.6f}")

def departure(values, min_count):
    """Observed minus model-expected positives per distinct value, in sigma."""
    u, inv, cnt = np.unique(values, return_inverse=True, return_counts=True)
    obs = np.bincount(inv, weights=yf)
    exp = np.bincount(inv, weights=P)
    var = np.bincount(inv, weights=P * (1 - P))
    z = (obs - exp) / np.sqrt(np.maximum(var, 1e-9))
    return u, cnt, obs, exp, z, cnt >= min_count

U, CNT, OBS, EXP, Z, TESTABLE = departure(RAW[CARRIER], CFG["MIN_VALUE_COUNT"])
N_TESTABLE = int(TESTABLE.sum())
print()
print(f"{CARRIER}: {len(U):,} distinct values, {N_TESTABLE:,} of them with at least "
      f"{CFG['MIN_VALUE_COUNT']} rows and therefore testable.")
print()
print(f"  {'|z| >':>7}{'values':>9}{'expected by chance':>21}")
SPIKE_COUNTS = {}
for thr in CFG["Z_THRESHOLDS"]:
    n = int((np.abs(Z) > thr)[TESTABLE].sum())
    SPIKE_COUNTS[thr] = n
    print(f"  {thr:>7}{n:>9}{2 * norm.sf(thr) * N_TESTABLE:>21.2f}")
print()
CONTROL_COL = max(NARROW, key=lambda c: CARD[c])
uc, cc_, oc, ec, zc, tc = departure(train[CONTROL_COL].astype(str).to_numpy(), 1)
print(f"  CONTROL, the same test on {CONTROL_COL} ({len(uc)} levels, which the model")
print(f"  already sees at full resolution): {int((np.abs(zc) > 3).sum())} values past "
      f"|z| > 3, largest |z| {np.abs(zc).max():.2f}")
print()
print("  The control returns nothing, as it must: a column the model already prices at full")
print("  resolution has no residual left in it. Income does, in quantity.")

### Family A: single-value spikes

In [13]:
order = np.argsort(-np.abs(np.where(TESTABLE, Z, 0)))
SPIKES = []
for i in order[:CFG["PRICE_TOP_N"]]:
    m = RAW[CARRIER] == U[i]
    SPIKES.append({"value": float(U[i]), "rows": int(CNT[i]), "rate": OBS[i] / CNT[i],
                   "expected": EXP[i] / CNT[i], "z": float(Z[i]),
                   "gain": refine_price(m)})
print(f"{'income value':>13}{'rows':>8}{'obs rate':>10}{'model expects':>15}{'z':>8}"
      f"{'AUC as a flag':>16}{'SE':>7}")
print("-" * 77)
for s in SPIKES:
    print(f"{s['value']:>13,.0f}{s['rows']:>8,}{s['rate']:>10.4f}{s['expected']:>15.4f}"
          f"{s['z']:>8.1f}{s['gain']:>+16.6f}{in_se(s['gain']):>7.2f}")
print("-" * 77)
print(f"  candidates searched in this family: {N_TESTABLE:,}")
print(f"  strongest single spike is worth {max(s['gain'] for s in SPIKES):+.6f} "
      f"({in_se(max(s['gain'] for s in SPIKES)):+.2f} SE) as a flag.")

### Families B and C: cliffs and zones

In [14]:
SUPPORT = U
cum_obs = np.concatenate([[0.0], np.cumsum(OBS)])
cum_exp = np.concatenate([[0.0], np.cumsum(EXP)])
cum_var = np.concatenate([[0.0], np.cumsum(np.bincount(
    np.unique(RAW[CARRIER], return_inverse=True)[1], weights=P * (1 - P)))])
cum_cnt = np.concatenate([[0.0], np.cumsum(CNT.astype(float))])

def block_z(a, b):
    """departure of support[a:b] from what the rest of the frame predicts, in sigma."""
    o = cum_obs[b] - cum_obs[a]; e = cum_exp[b] - cum_exp[a]; v = cum_var[b] - cum_var[a]
    return (o - e) / math.sqrt(max(v, 1e-9)), int(cum_cnt[b] - cum_cnt[a]), o, e

n_sup = len(SUPPORT)
CLIFFS = []
for a in range(CFG["CLIFF_STRIDE"], n_sup, CFG["CLIFF_STRIDE"]):
    z_, n_, o_, e_ = block_z(a, n_sup)
    if n_ >= CFG["MIN_VALUE_COUNT"]:
        CLIFFS.append({"thr": float(SUPPORT[a]), "z": z_, "rows": n_,
                       "rate": o_ / n_, "expected": e_ / n_})
N_CLIFF = len(CLIFFS)
CLIFFS.sort(key=lambda r: -abs(r["z"]))
for r in CLIFFS[:CFG["PRICE_TOP_N"]]:
    r["gain"] = refine_price(RAW[CARRIER] >= r["thr"])

ZONES = []
for a in range(0, n_sup - CFG["ZONE_WIDTH"], CFG["ZONE_STRIDE"]):
    b = a + CFG["ZONE_WIDTH"]
    z_, n_, o_, e_ = block_z(a, b)
    if n_ >= CFG["MIN_VALUE_COUNT"]:
        ZONES.append({"lo": float(SUPPORT[a]), "hi": float(SUPPORT[b - 1]), "z": z_,
                      "rows": n_, "rate": o_ / n_, "expected": e_ / n_})
N_ZONE = len(ZONES)
ZONES.sort(key=lambda r: -abs(r["z"]))
for r in ZONES[:CFG["PRICE_TOP_N"]]:
    r["gain"] = refine_price((RAW[CARRIER] >= r["lo"]) & (RAW[CARRIER] <= r["hi"]))

print(f"FAMILY B, cliffs: every {CFG['CLIFF_STRIDE']}th support value tried as a threshold, "
      f"{N_CLIFF} candidates")
print(f"{'income >=':>13}{'rows':>9}{'obs rate':>10}{'expects':>10}{'z':>9}"
      f"{'AUC as a flag':>16}{'SE':>7}")
print("-" * 74)
for r in CLIFFS[:CFG["PRICE_TOP_N"]]:
    print(f"{r['thr']:>13,.0f}{r['rows']:>9,}{r['rate']:>10.4f}{r['expected']:>10.4f}"
          f"{r['z']:>9.1f}{r['gain']:>+16.6f}{in_se(r['gain']):>7.2f}")
print()
print(f"FAMILY C, zones: sliding windows of {CFG['ZONE_WIDTH']} support values, "
      f"stride {CFG['ZONE_STRIDE']}, {N_ZONE} candidates")
print(f"{'from':>10}{'to':>10}{'rows':>9}{'obs rate':>10}{'expects':>10}{'z':>9}"
      f"{'AUC as a flag':>16}{'SE':>7}")
print("-" * 83)
for r in ZONES[:CFG["PRICE_TOP_N"]]:
    print(f"{r['lo']:>10,.0f}{r['hi']:>10,.0f}{r['rows']:>9,}{r['rate']:>10.4f}"
          f"{r['expected']:>10.4f}{r['z']:>9.1f}{r['gain']:>+16.6f}{in_se(r['gain']):>7.2f}")

### Families D and E: rounding lattices and truncation

In [15]:
iv = RAW[CARRIER]
is_int = np.allclose(iv, np.round(iv))
print(f"{CARRIER} is integer-valued: {is_int}. "
      f"Fractional parts present: {int((iv % 1 != 0).sum()):,}")
print()
print(f"FAMILY D, rounding lattices ({len(CFG['LATTICE_MODULI'])} candidates). A real income "
      f"column heaps hard on")
print("multiples of 1000. If this one does not, that is a fact about how it was made.")
print()
print(f"{'divisible by':>13}{'rows':>10}{'share':>9}{'if uniform':>12}{'ratio':>8}"
      f"{'AUC as a flag':>16}{'SE':>7}")
print("-" * 75)
LATTICE = []
for mod in CFG["LATTICE_MODULI"]:
    m = (iv % mod == 0)
    share = float(m.mean())
    g = refine_price(m)
    LATTICE.append({"mod": mod, "rows": int(m.sum()), "share": share,
                    "ratio": share * mod, "gain": g})
    print(f"{mod:>13,}{int(m.sum()):>10,}{share:>9.4f}{1 / mod:>12.4f}{share * mod:>8.2f}"
          f"{g:>+16.6f}{in_se(g):>7.2f}")
print("-" * 75)
CLAMP_LO = float(iv.min())
m_lo = iv == CLAMP_LO
rest = iv[iv > CLAMP_LO]
print()
print(f"FAMILY E, truncation at the range ends (4 candidates, both wide columns):")
print(f"{'flag':<38}{'rows':>9}{'share':>9}{'obs rate':>10}{'AUC as a flag':>16}{'SE':>7}")
print("-" * 89)
TRUNC = []
for c in WIDE:
    v = RAW[c]
    for lbl, m in ((f"{c} == min ({v.min():,.4g})", v == v.min()),
                   (f"{c} == max ({v.max():,.4g})", v == v.max())):
        g = refine_price(m) if c == CARRIER else float("nan")
        if c != CARRIER:
            k, cc = compress(qbins(RAW[c], CFG["REF_BINS"])[0] * 2 + m.astype(np.int64))
            s = BASE_NARROW + REF_LOGIT[CARRIER] + logit(crossfit(k, cc, CFG["REF_SMOOTH"]))
            g = roc_auc_score(y, s) - ADD_AUC
        TRUNC.append({"label": lbl, "rows": int(m.sum()), "gain": g})
        print(f"{lbl:<38}{int(m.sum()):>9,}{m.mean():>9.4f}{yf[m].mean():>10.4f}"
              f"{g:>+16.6f}{in_se(g):>7.2f}")
print("-" * 89)
print()
print(f"  The bottom of {CARRIER} is a hard clamp: {int(m_lo.sum()):,} rows "
      f"({m_lo.mean():.2%} of the frame) sit on")
print(f"  exactly {CLAMP_LO:,.0f}, and the next value up is {rest.min():,.0f}. That is not a "
      f"distribution with a floor,")
print(f"  that is a distribution that was clipped and had the clipped mass stacked on the "
      f"edge.")

### The whole census, ranked

In [16]:
CENSUS = []
for s in SPIKES:
    CENSUS.append(("spike", f"income == {s['value']:,.0f}", s["rows"], s["gain"]))
for r in CLIFFS[:CFG["PRICE_TOP_N"]]:
    CENSUS.append(("cliff", f"income >= {r['thr']:,.0f}", r["rows"], r["gain"]))
for r in ZONES[:CFG["PRICE_TOP_N"]]:
    CENSUS.append(("zone", f"income in [{r['lo']:,.0f}, {r['hi']:,.0f}]", r["rows"], r["gain"]))
for r in LATTICE:
    CENSUS.append(("lattice", f"income % {r['mod']:,} == 0", r["rows"], r["gain"]))
for r in TRUNC:
    CENSUS.append(("truncation", r["label"], r["rows"], r["gain"]))
CENSUS.sort(key=lambda r: -r[3])
N_SEARCHED = N_TESTABLE + N_CLIFF + N_ZONE + len(CFG["LATTICE_MODULI"]) + len(TRUNC)
N_PRICED = len(CENSUS)

print(f"Every candidate I priced, best first. {N_SEARCHED:,} candidates were searched across "
      f"five families;")
print(f"{N_PRICED} of them were carried through to a full AUC price.")
print()
print(f"{'rank':>5}{'family':>12}  {'flag':<40}{'rows':>9}{'AUC':>12}{'SE':>7}")
print("-" * 86)
for i, (fam, lbl, rows, g) in enumerate(CENSUS, 1):
    print(f"{i:>5}{fam:>12}  {lbl:<40}{rows:>9,}{g:>+12.6f}{in_se(g):>7.2f}")
print("-" * 86)
BEST_FLAG = CENSUS[0]
N_OVER_BAR = sum(1 for r in CENSUS if in_se(r[3]) >= 0.2)
print(f"  best flag on the board: {BEST_FLAG[1]} at {BEST_FLAG[3]:+.6f} "
      f"({in_se(BEST_FLAG[3]):+.2f} SE)")
print(f"  flags clearing the 0.2 SE bar: {N_OVER_BAR} of {N_PRICED}")
print(f"  the refinement null ceiling was {NULL_MAX:+.6f} ({in_se(NULL_MAX):+.2f} SE), so every")
print(f"  positive entry above is above the null. Being above the null is not the same as")
print(f"  being worth building, and this table is the difference between those two things.")
print()
if N_OVER_BAR == 0:
    print("  Not one candidate in any family clears 0.2 SE.")
else:
    print(f"  {N_OVER_BAR} candidate of {N_PRICED} reaches 0.2 SE, at {in_se(BEST_FLAG[3]):.2f}, and it is")
    print(f"  the maximum over {N_SEARCHED:,} searched candidates. That is what the largest of")
    print(f"  several thousand tests looks like when nothing real is there, and I am not going")
    print(f"  to promote it to a finding on the strength of clearing a bar by 0.01 SE.")
print()
print("This is the result I did not expect and it is the useful one. The artifacts are")
print("enormous in the data and worth nothing as features, because the reference partition")
print(f"already isolates them. A {CFG['REF_BINS']}-bin quantile cut does not split ties, so the "
      f"clamp mass lands")
print("in its own bin without being asked. You are already flagging it. You just did not know.")

---

## Part 7: The three inherited leads, each with its own number

Earlier campaign work recorded three specific artifacts in this column, used as hand flags by
a public notebook author here. I did not take them on faith and I did not want to bury them in
the ranked table, because a lead that turns out to be half right is more useful than one that
is repeated. Each gets recomputed from scratch, with the number it is worth.

In [17]:
LEAD_RESULTS = []
for label, kind, param in CFG["LEADS"]:
    print("=" * 92)
    print(label)
    print("-" * 92)
    if kind == "value":
        m = iv == param
        n = int(m.sum())
        nxt = iv[iv > param].min() if (iv > param).any() else float("nan")
        verdict = "HOLDS" if n > 0 and abs(yf[m].mean() - yf[~m].mean()) > 0.05 else "DOES NOT HOLD"
        print(f"  rows at exactly {param:,}      {n:>10,}   ({m.mean():.2%} of the frame)")
        print(f"  target rate there             {yf[m].mean():>10.6f}")
        print(f"  target rate everywhere else   {yf[~m].mean():>10.6f}")
        print(f"  next value above it           {nxt:>10,.0f}")
        print(f"  it is the column minimum      {str(param == iv.min()):>10}")
        g = refine_price(m)
    elif kind == "cliff":
        m = iv >= param
        n = int(m.sum())
        max_neg = float(iv[y == 0].max())
        first_above = float(iv[iv > max_neg].min()) if (iv > max_neg).any() else float("nan")
        n_above = int((iv > max_neg).sum())
        verdict = "HOLDS" if n > 0 and yf[m].mean() > 0.95 else "DOES NOT HOLD"
        print(f"  rows at or above {param:,}     {n:>10,}")
        print(f"  target rate there             {yf[m].mean():>10.6f}")
        print(f"  target rate below it          {yf[~m].mean():>10.6f}")
        print(f"  highest income among NEGATIVES{max_neg:>10,.0f}")
        print(f"  lowest value above that       {first_above:>10,.0f}   ({n_above:,} rows, "
              f"all positive)")
        print(f"  so the true boundary is somewhere in ({max_neg:,.0f}, {first_above:,.0f}]; "
              f"{param:,} is the")
        print(f"  first observed value above it, not a special number in the generator.")
        print(f"  probability of {n_above} straight positives if the local rate were "
              f"{float(yf[(iv > max_neg - 20000) & (iv <= max_neg)].mean()):.3f}: "
              f"{float(yf[(iv > max_neg - 20000) & (iv <= max_neg)].mean()) ** n_above:.3g}")
        g = refine_price(m)
    else:
        lo, hi = param
        m = (iv >= lo) & (iv <= hi)
        n = int(m.sum())
        gap_lo = float(iv[iv > iv.min()].min())
        print(f"  rows in [{lo:,}, {hi:,}]      {n:>10,}")
        print(f"  target rate there             "
              f"{(yf[m].mean() if n else float('nan')):>10.6f}")
        print(f"  target rate everywhere else   {yf[~m].mean():>10.6f}")
        print(f"  rows strictly between the clamp at {iv.min():,.0f} and {lo:,}: "
              f"{int(((iv > iv.min()) & (iv < lo)).sum()):,}")
        print(f"  first value above the clamp   {gap_lo:>10,.0f}")
        srt = np.unique(iv)
        d = np.diff(srt)
        j = int(np.argmax(d))
        print(f"  widest gap in the whole support: {srt[j]:,.0f} to {srt[j + 1]:,.0f}, "
              f"a span of {d[j]:,.0f} with nothing in it")
        dense = n > 0
        flat = (n > 0) and (yf[m].mean() < 0.02)
        verdict = ("HALF RIGHT" if dense and flat else
                   "HOLDS" if flat else "DOES NOT HOLD")
        print(f"  the named window is NOT empty ({n:,} rows in it), so it is not a dead zone")
        print(f"  in the sense of missing data. Its target rate is near zero, so it IS a dead")
        print(f"  zone in the sense of a flat response. The genuinely empty stretch is the")
        print(f"  one above, and it is not where the lead pointed.")
        g = refine_price(m)
    LEAD_RESULTS.append({"label": label, "verdict": verdict, "rows": int(m.sum()),
                         "gain": float(g)})
    print(f"  VERDICT: {verdict}")
    print(f"  worth as a flag: {g:+.6f}  ({in_se(g):+.2f} SE)   "
          f"{'above' if in_se(g) >= 0.2 else 'BELOW'} the 0.2 SE bar")
print("=" * 92)
print()
print(f"{'lead':<46}{'verdict':>13}{'rows':>9}{'AUC':>12}{'SE':>7}")
print("-" * 87)
for r in LEAD_RESULTS:
    print(f"{r['label']:<46}{r['verdict']:>13}{r['rows']:>9,}{r['gain']:>+12.6f}"
          f"{in_se(r['gain']):>7.2f}")
print("-" * 87)
print(f"  {sum(1 for r in LEAD_RESULTS if r['verdict'] == 'HOLDS')} of "
      f"{len(LEAD_RESULTS)} hold as stated, "
      f"{sum(1 for r in LEAD_RESULTS if r['verdict'] == 'HALF RIGHT')} is half right.")
print(f"  {sum(1 for r in LEAD_RESULTS if in_se(r['gain']) >= 0.2)} of "
      f"{len(LEAD_RESULTS)} are worth building.")

---

## Part 8: So what does collect the resolution gain

If no flag is worth building and the gain is real, the gain must be spread over many values at
once. That is testable: take the values ranked by departure and split out the top k as their
own cells, for a rising k, and watch the gain accumulate.

In [18]:
KMAX = [1, 3, 10, 30, 100, 300, 1000, 3000]
ACCUM = []
inv_all = np.unique(RAW[CARRIER], return_inverse=True)[1]
for kk in KMAX:
    sel = np.zeros(len(U), dtype=bool)
    sel[order[:kk]] = True
    m = sel[inv_all]
    k_grp, c_grp = compress(K_REF * 2 + m.astype(np.int64))
    g_grp = additive_with(CARRIER, k_grp, c_grp, CFG["REF_SMOOTH"]) - ADD_AUC
    k_own, c_own = compress(np.where(m, inv_all + CFG["REF_BINS"] + 1, K_REF))
    g_own = additive_with(CARRIER, k_own, c_own, CFG["REF_SMOOTH"]) - ADD_AUC
    ACCUM.append({"k": kk, "rows": int(m.sum()), "cells": c_own,
                  "grouped": g_grp, "own": g_own})

print(f"Top-k income values by departure, added to the {CFG['REF_BINS']}-bin reference "
      f"partition:")
print()
print(f"{'k':>6}{'rows':>9}{'as ONE flag':>14}{'SE':>7}{'as own cells':>15}{'SE':>7}"
      f"{'cells':>8}")
print("-" * 66)
for r in ACCUM:
    print(f"{r['k']:>6}{r['rows']:>9,}{r['grouped']:>+14.6f}{in_se(r['grouped']):>7.2f}"
          f"{r['own']:>+15.6f}{in_se(r['own']):>7.2f}{r['cells']:>8}")
print("-" * 66)
BESTK = max(ACCUM, key=lambda r: r["own"])
print(f"  best of these: k={BESTK['k']} split into their own cells, "
      f"{BESTK['own']:+.6f} ({in_se(BESTK['own']):+.2f} SE)")
print(f"  the whole fine partition from Part 4:   "
      f"{SOLO[CARRIER]:+.6f} ({in_se(SOLO[CARRIER]):+.2f} SE)")
print()
print("Two things are visible here and both matter.")
print()
print("One: collapsing many values into a single flag is strictly worse than giving each its")
print("own cell, and the gap widens with k. That is the signature of values that disagree")
print("with each other, not of a shared property they all have. A flag says these rows are")
print("special in the same way. They are not.")
print()
print("Two: it takes hundreds of values to collect a gain the binning collects for free. The")
print("top ten are worth a fraction of a standard error between them. There is no short list.")

---

## Part 9: What this says about the generator, and what it does not

I want to be careful here, because this is the part where a measurement turns into a story and
the story can run ahead of the number. So: measured first, inferred second, and labelled.

**Measured.** The column is integer-valued with no fractional parts. It has a hard floor: a
large share of the frame sits on exactly the minimum value and the next value up is a long way
above it. It does not heap on round numbers; multiples of 1000 are rarer than chance, not
commoner. Its support is a set of some thousands of specific integers with uneven spacing and
very uneven multiplicity, some values carrying hundreds of rows and thousands carrying one or
two. Dozens of individual values depart from what the rest of the frame predicts for them by
more than five sigma, against zero expected across the same number of tests. The same test on
a narrow column returns nothing. Above a certain income every single row in train is a
positive, with no negatives anywhere above it.

**Inferred, and this is inference.** A column drawn from a smooth parametric distribution does
not look like this. A column copied from a real survey does not look like this either, because
real income data heaps enormously on multiples of 1000 and this does not. What does look like
this is a generator that learned the *empirical support* of an original column and resamples
from it, attaching a propensity to each value rather than to a smooth function of the value.
The per-value departures are then not noise and not a signal about income as a quantity. They
are the generator's memory of which specific rows in the original frame were positives.

Three checks that support that reading, all of them in cells above or below: the departures
survive calibration against the other twelve columns; they are absent from a narrow column
tested identically; and grouping departing values into one flag is worse than letting each
keep its own cell, which is what you would expect if the values share no property except
having been memorised separately.

**What I am not claiming.** I did not see the generating code and neither did you. "The
generator memorised the support" is a description of how this data behaves under two
instruments, not a proof about anyone's script. A different mechanism could produce the same
fingerprints. The floor clamp in particular is equally consistent with the original dataset
having been collected with an income floor and no synthesis involved at all, and I cannot
distinguish those from inside the frame.

The check below is the one piece of direct evidence about resampling I can run without leaving
the competition data, and it cuts both ways, so it goes in the notebook rather than in a
footnote.

In [19]:
tr_vals = set(np.unique(RAW[CARRIER]).tolist())
te_vals = np.unique(test[CARRIER].to_numpy(float))
new = [v for v in te_vals if v not in tr_vals]
print(f"{CARRIER} support, train against test:")
print(f"  distinct values in train      {len(tr_vals):>8,}")
print(f"  distinct values in test       {len(te_vals):>8,}")
print(f"  in test but not in train      {len(new):>8,}   "
      f"({100 * len(new) / len(te_vals):.1f}% of the test support)")
te_raw = test[CARRIER].to_numpy(float)
rows_new = int(np.isin(te_raw, new).sum())
print(f"  test ROWS on a value train never saw   {rows_new:>8,}   "
      f"({100 * rows_new / len(te_raw):.2f}% of test rows)")
print()
print("Read this honestly. If the generator resampled a fixed finite support, test would")
print("contain no value train has not seen, and it does contain some. So it is not pure")
print("resampling from a closed list. But the overlap is very high and the novel values are")
print("a small share of test rows, which is what a generator that mostly reproduces a")
print("learned support and occasionally interpolates between its points would give you.")
print()
print("The practical consequence is the one that matters for anyone fitting this: a")
print("value-level lookup on income degrades gracefully here, because the values it has")
print("never seen are rare and the smoothing sends them to the prior. That is why the fine")
print("partitions in Part 3 do not fall apart, and it is a fact about this test set that")
print("would not hold on a different one.")
print()
CNT_TR = CNT
print(f"Multiplicity in train: {int((CNT_TR == 1).sum()):,} values appear exactly once, "
      f"{int((CNT_TR >= 100).sum()):,} appear 100 times or more,")
print(f"the most common non-clamp value appears "
      f"{int(np.sort(CNT_TR[U != iv.min()])[-1]):,} times. Median rows per value: "
      f"{int(np.median(CNT_TR))}.")

---

## Part 10: What would make this wrong

**One split, one seed.** Every AUC here comes from a single `StratifiedKFold(5, shuffle=True,
random_state=42)` partition. I did not repeat it across seeds. The resolution result is several
standard errors and will survive a seed change. The ordering inside the census table will not,
and I would not defend any two adjacent rows of it against each other.

**Cross-validation gains are not leaderboard gains.** Every number here is out-of-fold on
train. The reference standard error is a statement about a public split I have never scored
against, under an assumed 20% split fraction that Kaggle does not publish. A gain of three
standard errors measured this way is a claim about this data and you should discount it before
believing it about a leaderboard.

**The census searched a lot of candidates, and the ranking has to be read that way.** The cell
prints the exact count: thousands of single values, hundreds of thresholds, hundreds of zones,
ten moduli and four truncation flags. A maximum taken over that many candidates is biased
upward, and the right null for the top of the table is not zero but the maximum of the same
number of null draws. I ran a refinement null at three mask sizes and it sits at essentially
zero, which is why I am willing to call the small positive entries real. I am not willing to
call the ordering among them real, and the whole point of the section is that none of them is
worth building anyway, so the multiple-comparisons exposure does not change a decision here.

**The departure test depends on a calibration.** The z-scores in Part 6 compare observed
positives against a sum of predicted probabilities that came from binning the additive score
into 200 quantile bins. A different calibrator would move the z-scores. It would not move the
count from dozens to zero, because the control column returns nothing under the identical
calibration, but the exact spike counts at each threshold are a property of that choice.

**"The generator did X" is an inference.** Part 9 says this at length and I will say it once
more in the limits section because it is the claim most likely to be repeated without its
qualifier. Everything about the generating process in this notebook is read off the data. None
of it is a statement about code anyone has seen.

**The refinement instrument has a bias I should name.** Pricing a flag by refining the
reference partition means a flag is worth what it adds *given that partition*. A flag that is
redundant with the partition prices at zero even if it is enormous in the data, and that is
exactly what happens to all three inherited leads. That is the finding, but it also means this
notebook cannot tell you what those flags would be worth to a model that does not bin income
at all. A linear model in raw income would find the clamp flag extremely valuable. I did not
measure that and the numbers here do not speak to it.

**Smoothing is swept but the sweep is coarse.** Five values, log-spaced-ish. The optimum on
the grid is an interior point, so I do not think a finer sweep changes the shape, but the exact
best AUC is the best of forty-five points on a grid and not an optimum.

**I only tested the two columns that can be binned.** Fine structure in a narrow column is not
a thing this notebook can find, by construction. The control in Part 6 suggests there is none
to find, but a control on one column is not a survey of twelve.

---

## Part 11: What to do with this

1. **Bin your wide numerics far finer than 32, and move the smoothing when you do.** This is
   the whole practical content. It is one line of feature engineering, it needs no new columns,
   and on this board it is worth more than the entire interaction structure I spent a previous
   notebook finding.
2. **Do not build the flags.** Not the clamp, not the cliff, not the zone. They are real, they
   are dramatic in a histogram, and a quantile partition already has them because quantile bins
   do not split ties. All three inherited leads price under a tenth of a standard error, and
   across five families and thousands of candidates the single best flag reaches 0.21 SE as the
   maximum of a large search, which is not a number I would build on.
3. **Before you build any flag, price it by refinement, not by addition.** Adding a redundant
   term to a summed-logit model makes a known artifact look strongly negative, and that reads
   like a discovery. Cross the flag into the partition instead and the double counting goes
   away.
4. **A departure test needs a control column.** Running it on income alone would have produced
   an impressive number with nothing to compare it to. Running the identical test on a column
   the model already sees at full resolution is what makes the income result mean something,
   and it costs one extra cell.

The cell to fork is `refine_price`. Point it at your own mask, keep the folds, and read the
delta against the null rather than against zero.

If something here does not reproduce for you, say so in the comments with the cell and the
number. I would rather correct this than be quietly wrong in it.

The decomposition this notebook follows up is in
[S6E9: 4 of 13 Columns Hold the Interaction](https://www.kaggle.com/code/dariushafshar/s6e9-4-of-13-columns-hold-the-interaction).
The variance result is DeLong, DeLong and Clarke-Pearson (1988).

In [20]:
HEAD = {
    "n_train": N_TRAIN, "n_test": N_TEST, "n_public_assumed": N_PUBLIC,
    "prevalence": round(PREV, 6),
    "se_reference": SE, "se_live_delong_public": round(SE_LIVE, 6),

    "additive_auc": round(float(ADD_AUC), 6),
    "additive_published": CFG["PUBLISHED_ADDITIVE"],
    "additive_reproduces": bool(abs(ADD_AUC - CFG["PUBLISHED_ADDITIVE"]) < 5e-6),

    "best_auc": round(float(BEST_AUC), 6),
    "best_bins": BEST_KEY[0], "best_smooth": BEST_KEY[1],
    "best_cells": {k: int(v) for k, v in BEST_CELLS.items()},
    "resolution_gain": round(float(RES_GAIN), 6),
    "resolution_gain_in_se": round(in_se(RES_GAIN), 2),
    "fixed_smooth50_best_bins": FIXED50_BEST[0],
    "fixed_smooth50_best_auc": round(float(FIXED50_BEST[1]), 6),
    "share_missed_by_fixed_smoothing": round(
        100 * (BEST_AUC - FIXED50_BEST[1]) / RES_GAIN, 1),

    "carrier_column": CARRIER,
    "carrier_gain": round(float(SOLO[CARRIER]), 6),
    "carrier_share_pct": round(100 * SOLO[CARRIER] / TOT, 1),
    "other_wide_column": OTHER,
    "other_gain": round(float(SOLO[OTHER]), 6),

    "null_within_bin_mean": round(float(nulls.mean()), 6),
    "null_within_bin_max": round(float(nulls.max()), 6),
    "real_minus_null": round(float(SOLO_REAL - nulls.max()), 6),
    "regions": [[r["g"], round(r["gain"], 6), round(in_se(r["gain"]), 2)] for r in REGION],
    "region_sum": round(float(SUM_R), 6), "region_joint": round(float(JOINT), 6),

    "testable_values": N_TESTABLE,
    "spike_counts_by_z": {str(k): v for k, v in SPIKE_COUNTS.items()},
    "control_column": CONTROL_COL,
    "control_values_past_z3": int((np.abs(zc) > 3).sum()),
    "control_max_abs_z": round(float(np.abs(zc).max()), 2),

    "candidates_searched": int(N_SEARCHED),
    "candidates_priced": int(N_PRICED),
    "refinement_null_max": round(float(NULL_MAX), 6),
    "best_flag": [BEST_FLAG[1], round(float(BEST_FLAG[3]), 6), round(in_se(BEST_FLAG[3]), 2)],
    "flags_over_0p2_se": int(N_OVER_BAR),
    "census_top": [[f, l, n, round(g, 6), round(in_se(g), 2)] for f, l, n, g in CENSUS[:10]],

    "leads": [[r["label"], r["verdict"], r["rows"], round(r["gain"], 6),
               round(in_se(r["gain"]), 2)] for r in LEAD_RESULTS],
    "leads_holding": sum(1 for r in LEAD_RESULTS if r["verdict"] == "HOLDS"),
    "leads_worth_building": sum(1 for r in LEAD_RESULTS if in_se(r["gain"]) >= 0.2),

    "accumulation": [[r["k"], round(r["grouped"], 6), round(r["own"], 6)] for r in ACCUM],
    "test_support_novel_values": int(len(new)),
    "test_rows_on_novel_values": int(rows_new),

    "smoke_rows": SMOKE_ROWS,
    "wall_seconds": round(time.time() - T0, 1),
}
os.makedirs(WORK, exist_ok=True)
with open(os.path.join(WORK, "headline.json"), "w") as fh:
    json.dump(HEAD, fh, indent=1)
np.save(os.path.join(WORK, "folds_seed42.npy"), FOLDS)
np.save(os.path.join(WORK, "additive_logit_oof.npy"), ADDITIVE.astype(np.float32))
for k, v in HEAD.items():
    print(f"  {k:<32} {v}")
print()
print("written: headline.json, folds_seed42.npy, additive_logit_oof.npy")
print()
print(f"TITLE CHECK: {SPIKE_COUNTS[5]} spikes past |z|>5 in {CARRIER}, "
      f"{N_OVER_BAR} of {N_PRICED} priced flags clear 0.2 SE.")
if SMOKE_ROWS:
    print("SMOKE_ROWS was set. Do NOT freeze a title from this run.")